# Hˢ — Higgins Decomposition Dashboard

**A compositional inference instrument operating on the simplex.**

Load any CSV of compositions → full 12-step pipeline → diagnostic codes → structural modes → investigation prompts.

CoDa training is sufficient. No domain-specific configuration needed.

---

### Requirements
```
pip install numpy matplotlib pandas
```

### Usage
1. Set `DATA_FILE` to your CSV path (or use the built-in demo data)
2. Run All Cells
3. Read the structural mode investigation prompts at the bottom

---

In [ ]:
# ════════════════════════════════════════════════════════════
# CONFIGURATION — Edit this cell
# ════════════════════════════════════════════════════════════

# Point to your CSV file (columns = carriers, rows = observations)
# Set to None to use built-in Nuclear SEMF demo data
DATA_FILE = None  # e.g., "mydata.csv", "geochemistry.csv"

# Optional: override auto-detected settings
SYSTEM_NAME = None      # e.g., "Major Oxides"
DOMAIN = None            # e.g., "GEOCHEMISTRY"
REPORT_LANGUAGE = "en"   # en, zh, hi, pt, it

# Dashboard options
SHOW_METROLOGY = True    # Run instrument self-evaluation
FIGURE_DPI = 100         # Plot resolution

In [ ]:
# ════════════════════════════════════════════════════════════
# SETUP — Load pipeline (run once)
# ════════════════════════════════════════════════════════════

import sys, os, json
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('agg')  # safe backend
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
from IPython.display import display, HTML, Markdown

# Find pipeline directory
NOTEBOOK_DIR = os.path.dirname(os.path.abspath('__file__'))
PIPELINE_DIR = os.path.join(NOTEBOOK_DIR, 'pipeline') if os.path.exists(os.path.join(NOTEBOOK_DIR, 'pipeline')) else os.path.join(NOTEBOOK_DIR, 'tools', 'pipeline')
if not os.path.exists(PIPELINE_DIR):
    # Try relative to repo root
    for candidate in ['tools/pipeline', '../tools/pipeline', '../../tools/pipeline']:
        if os.path.exists(candidate):
            PIPELINE_DIR = os.path.abspath(candidate)
            break

sys.path.insert(0, PIPELINE_DIR)

from higgins_decomposition_12step import HigginsDecomposition, NumpyEncoder
from hs_codes import generate_codes, CODE_DICTIONARY, STRUCTURAL_MODES
from hs_reporter import report

print(f"Pipeline: {PIPELINE_DIR}")
print(f"Codes: {len(CODE_DICTIONARY)} defined, {len(STRUCTURAL_MODES)} structural modes")
print(f"Ready.")

In [ ]:
# ════════════════════════════════════════════════════════════
# LOAD DATA
# ════════════════════════════════════════════════════════════

if DATA_FILE and os.path.exists(DATA_FILE):
    df = pd.read_csv(DATA_FILE)
    # Drop non-numeric columns (index, labels, etc.)
    numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    carriers = numeric_cols
    data = df[carriers].values
    source_desc = f"Loaded from {os.path.basename(DATA_FILE)}"
    exp_name = SYSTEM_NAME or os.path.splitext(os.path.basename(DATA_FILE))[0].replace('_', ' ').title()
    exp_domain = DOMAIN or "USER_DATA"
    print(f"Loaded: {DATA_FILE}")
else:
    # Built-in demo: Nuclear SEMF (Z=1 to 92)
    aV, aS, aC, aA, aP = 15.56, 17.23, 0.7, 23.285, 12.0
    rows = []
    for Z in range(1, 93):
        A = round(2.5 * Z)
        vol = aV * A
        surf = aS * A**(2/3)
        coul = aC * Z * (Z - 1) / A**(1/3)
        asym = aA * (A - 2*Z)**2 / A
        pair = 0
        if Z % 2 == 0 and (A - Z) % 2 == 0: pair = aP / A**0.5
        elif Z % 2 == 1 and (A - Z) % 2 == 1: pair = -aP / A**0.5
        p1, p2, p3 = abs(vol), abs(surf) + abs(coul), abs(asym) + abs(pair)
        rows.append([p1, p2, p3])
    data = np.array(rows)
    carriers = ['Volume', 'Surf+Coul', 'Sym+Pair']
    source_desc = "SEMF binding energy (Weizsäcker), Z=1-92"
    exp_name = SYSTEM_NAME or "Nuclear SEMF"
    exp_domain = DOMAIN or "NUCLEAR"
    print("Using built-in demo: Nuclear SEMF (Z=1→92)")

N, D = data.shape
print(f"N={N} observations, D={D} carriers")
print(f"Carriers: {', '.join(carriers)}")

# Show data preview
df_preview = pd.DataFrame(data[:8], columns=carriers)
display(df_preview.style.set_caption(f"First 8 of {N} observations").format('{:.4f}'))

In [ ]:
# ════════════════════════════════════════════════════════════
# RUN PIPELINE
# ════════════════════════════════════════════════════════════

hd = HigginsDecomposition(
    experiment_id="DASH",
    name=exp_name,
    domain=exp_domain,
    carriers=carriers,
    data_source_type="LOCAL FILE" if DATA_FILE else "DERIVED",
    data_source_description=source_desc
)
hd.load_data(data)
result = hd.run_full_extended()

codes = generate_codes(result)
sm_codes = [c for c in codes if c['code'].startswith('SM-')]

# Classification
classification = 'NATURAL' if any(c['code'] == 'S8-NAT-INF' for c in codes) \
    else 'INVESTIGATE' if any(c['code'] == 'S8-INV-WRN' for c in codes) else 'FLAG'
cls_color = {'NATURAL': '#3FB950', 'INVESTIGATE': '#F0883E', 'FLAG': '#F85149'}[classification]

# Key metrics
s = result['steps']
hvld_shape = s.get('step6_pll_shape', '?')
hvld_r2 = s.get('step6_pll_R2', 0)
sq = s.get('step7_squeeze_closest')
nearest_const = sq.get('constant', '—') if sq else '—'
nearest_delta = sq.get('delta', 0) if sq else 0
eitt = s.get('step8_eitt_invariance', {})
eitt_pass = all(v.get('pass', False) for v in eitt.values()) if eitt else False

# Display summary card
display(HTML(f"""
<div style='background:#161B22;color:#E6EDF3;padding:16px;border-radius:8px;border:2px solid {cls_color};font-family:Consolas,monospace;margin:8px 0'>
  <div style='font-size:20px;color:#FFD700;font-weight:bold;margin-bottom:8px'>Hˢ Pipeline Results</div>
  <table style='color:#E6EDF3;font-size:14px;border-collapse:collapse'>
    <tr><td style='padding:3px 16px 3px 0;color:#8B949E'>System</td><td style='font-weight:bold'>{exp_name}</td></tr>
    <tr><td style='padding:3px 16px 3px 0;color:#8B949E'>Domain</td><td>{exp_domain}</td></tr>
    <tr><td style='padding:3px 16px 3px 0;color:#8B949E'>N × D</td><td>{N} × {D}</td></tr>
    <tr><td style='padding:3px 16px 3px 0;color:#8B949E'>Classification</td><td style='color:{cls_color};font-weight:bold;font-size:16px'>{classification}</td></tr>
    <tr><td style='padding:3px 16px 3px 0;color:#8B949E'>HVLD</td><td>{hvld_shape} (R² = {hvld_r2:.4f})</td></tr>
    <tr><td style='padding:3px 16px 3px 0;color:#8B949E'>Nearest constant</td><td style='color:#FFD700'>{nearest_const} (δ = {nearest_delta:.6f})</td></tr>
    <tr><td style='padding:3px 16px 3px 0;color:#8B949E'>EITT</td><td style='color:{"#3FB950" if eitt_pass else "#F0883E"}'>{"PASS" if eitt_pass else "FAIL"}</td></tr>
    <tr><td style='padding:3px 16px 3px 0;color:#8B949E'>Codes</td><td>{len(codes)} ({len(sm_codes)} structural modes)</td></tr>
  </table>
</div>
"""))

In [ ]:
# ════════════════════════════════════════════════════════════
# PANEL 1: σ²_A Variance Trajectory + HVLD
# ════════════════════════════════════════════════════════════

fig, axes = plt.subplots(2, 3, figsize=(18, 10), facecolor='#0D1117')
for ax in axes.flat:
    ax.set_facecolor('#161B22')
    ax.tick_params(colors='#8B949E', labelsize=8)
    for spine in ax.spines.values():
        spine.set_color('#30363D')

# ── Panel 1: Variance trajectory ──
ax1 = axes[0, 0]
sig2 = s.get('step5_sigma2_trajectory', [])
if sig2:
    ax1.plot(sig2, color='#58A6FF', linewidth=1.5)
    if sq:
        const_val = sq.get('constant_value', 0)
        if const_val and const_val < max(sig2) * 1.2:
            ax1.axhline(y=const_val, color='#FFD700', linestyle='--', linewidth=1, alpha=0.7)
            ax1.text(len(sig2)*0.02, const_val*1.05, f"{nearest_const} = {const_val:.5f}",
                     color='#FFD700', fontsize=8)
ax1.set_title('σ²_A Variance Trajectory', color='#FFD700', fontsize=10, fontweight='bold')
ax1.set_xlabel('Observation index', color='#8B949E', fontsize=8)
ax1.set_ylabel('σ²_A', color='#8B949E', fontsize=8)

# ── Panel 2: Composition bars (first, middle, last) ──
ax2 = axes[0, 1]
closed = np.array(s.get('step3_closed_data', data / data.sum(axis=1, keepdims=True)))
indices = [0, N//2, N-1]
colors_bar = ['#FFD700', '#39D2C0', '#F0883E', '#BC8CFF', '#58A6FF', '#F85149', '#3FB950']
bar_width = 0.25
for bi, idx in enumerate(indices):
    row = closed[idx]
    positions = np.arange(D) + bi * bar_width
    for j in range(D):
        ax2.bar(positions[j], row[j], bar_width * 0.9, color=colors_bar[j % len(colors_bar)], alpha=0.7)
ax2.set_xticks(np.arange(D) + bar_width)
ax2.set_xticklabels([c[:8] for c in carriers], fontsize=7, color='#8B949E', rotation=30)
ax2.set_title('Compositions (first / mid / last)', color='#FFD700', fontsize=10, fontweight='bold')
ax2.set_ylabel('Proportion', color='#8B949E', fontsize=8)

# ── Panel 3: Ternary (D=3) or CLR scatter (D>3) ──
ax3 = axes[0, 2]
clr_data = np.array(s.get('step4_clr_data', []))
if len(clr_data) > 0 and D == 3:
    # Ternary projection
    def to_ternary(row):
        ss = row[0] + row[1] + row[2]
        x = 0.5 * (2 * row[1] + row[2]) / ss
        y = (np.sqrt(3) / 2) * row[2] / ss
        return x, y
    pts = np.array([to_ternary(r) for r in closed])
    colors_pts = plt.cm.viridis(np.linspace(0, 1, N))
    ax3.scatter(pts[:, 0], pts[:, 1], c=colors_pts, s=20, zorder=3)
    ax3.plot(pts[:, 0], pts[:, 1], color='#58A6FF', alpha=0.3, linewidth=0.8)
    tri = np.array([[0,0],[1,0],[0.5,np.sqrt(3)/2],[0,0]])
    ax3.plot(tri[:, 0], tri[:, 1], color='#30363D', linewidth=0.5)
    ax3.set_title('Ternary Projection', color='#FFD700', fontsize=10, fontweight='bold')
elif len(clr_data) > 0:
    # CLR scatter for D>3
    ax3.scatter(clr_data[:, 0], clr_data[:, 1], c=np.arange(N), cmap='viridis', s=20)
    ax3.plot(clr_data[:, 0], clr_data[:, 1], color='#58A6FF', alpha=0.3, linewidth=0.8)
    ax3.axhline(y=0, color='#30363D', linewidth=0.5)
    ax3.axvline(x=0, color='#30363D', linewidth=0.5)
    ax3.set_xlabel(f'CLR({carriers[0][:8]})', color='#8B949E', fontsize=8)
    ax3.set_ylabel(f'CLR({carriers[1][:8]})', color='#8B949E', fontsize=8)
    ax3.set_title('CLR Projection', color='#FFD700', fontsize=10, fontweight='bold')
ax3.set_aspect('equal')

# ── Panel 4: Shannon Entropy ──
ax4 = axes[1, 0]
entropy = s.get('step8_entropy_trajectory', [])
if entropy:
    ax4.plot(entropy, color='#3FB950', linewidth=1.5)
    mean_h = np.mean(entropy)
    ax4.axhline(y=mean_h, color='#F85149', linestyle='--', linewidth=1, alpha=0.6)
    ax4.text(len(entropy)*0.02, mean_h + 0.02, f'H̄ = {mean_h:.4f}', color='#F85149', fontsize=8)
ax4.set_title(f'Shannon Entropy (EITT: {"PASS" if eitt_pass else "FAIL"})',
              color='#FFD700', fontsize=10, fontweight='bold')
ax4.set_xlabel('Observation index', color='#8B949E', fontsize=8)
ax4.set_ylabel('H (normalised)', color='#8B949E', fontsize=8)

# ── Panel 5: Complex plane ──
ax5 = axes[1, 1]
if len(clr_data) > 0:
    re_vals = clr_data[:, 0] - np.mean(clr_data[:, 0])
    im_vals = clr_data[:, 1] - np.mean(clr_data[:, 1])
    colors_cplx = plt.cm.plasma(np.linspace(0, 1, N))
    ax5.scatter(re_vals, im_vals, c=colors_cplx, s=20, zorder=3)
    ax5.plot(re_vals, im_vals, color='#BC8CFF', alpha=0.2, linewidth=0.8)
    ax5.axhline(y=0, color='#30363D', linewidth=0.5)
    ax5.axvline(x=0, color='#30363D', linewidth=0.5)
    for i in range(N):
        ax5.plot([0, re_vals[i]], [0, im_vals[i]], color='#FFD700', alpha=0.05, linewidth=0.5)
ax5.set_title('Complex Plane (CLR Phase)', color='#FFD700', fontsize=10, fontweight='bold')
ax5.set_xlabel('Re(CLR)', color='#8B949E', fontsize=8)
ax5.set_ylabel('Im(CLR)', color='#8B949E', fontsize=8)
ax5.set_aspect('equal')

# ── Panel 6: Per-carrier contribution ──
ax6 = axes[1, 2]
ext = result.get('extended', {}).get('universal', {})
pcc = ext.get('per_carrier_contribution', {})
contributions = pcc.get('contributions', {})
if contributions:
    carr_names = list(contributions.keys())
    carr_vals = [contributions[c] for c in carr_names]
    bars = ax6.barh(range(len(carr_names)), carr_vals,
                    color=[colors_bar[i % len(colors_bar)] for i in range(len(carr_names))], alpha=0.8)
    ax6.set_yticks(range(len(carr_names)))
    ax6.set_yticklabels([c[:12] for c in carr_names], fontsize=8, color='#E6EDF3')
    for i, v in enumerate(carr_vals):
        ax6.text(v + 0.5, i, f'{v:.1f}%', color='#E6EDF3', fontsize=8, va='center')
ax6.set_title('Per-Carrier Variance Contribution', color='#FFD700', fontsize=10, fontweight='bold')
ax6.set_xlabel('% of total variance', color='#8B949E', fontsize=8)

plt.suptitle(f'Hˢ — {exp_name} ({exp_domain}) — {classification}',
             color='#FFD700', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('Hs_dashboard.png', dpi=FIGURE_DPI, bbox_inches='tight', facecolor='#0D1117')
plt.show()
print(f"Dashboard saved: Hs_dashboard.png")

In [ ]:
# ════════════════════════════════════════════════════════════
# DIAGNOSTIC CODES TABLE
# ════════════════════════════════════════════════════════════

# Build styled table
rows_html = []
level_colors = {
    'INF': '#3FB950', 'WRN': '#F0883E', 'ERR': '#F85149',
    'DIS': '#58A6FF', 'CAL': '#BC8CFF'
}

# Pipeline codes (non-SM)
pipeline_codes = [c for c in codes if not c['code'].startswith('SM-')]
for c in pipeline_codes:
    level = c['code'].split('-')[-1]
    color = level_colors.get(level, '#8B949E')
    val = ''
    if c.get('value'):
        if isinstance(c['value'], dict):
            parts = [f"{k}={v:.6f}" if isinstance(v, float) else f"{k}={v}" for k, v in list(c['value'].items())[:3]]
            val = ', '.join(parts)
        elif isinstance(c['value'], (int, float)):
            val = str(c['value'])
    rows_html.append(
        f"<tr><td style='font-family:Consolas;color:{color};padding:2px 8px'>[{c['code']}]</td>"
        f"<td style='color:#E6EDF3;padding:2px 8px'>{c['short']}</td>"
        f"<td style='color:#8B949E;padding:2px 8px;font-size:11px'>{val}</td></tr>"
    )

table_html = f"""
<div style='background:#161B22;padding:12px;border-radius:8px;margin:8px 0'>
<div style='color:#FFD700;font-weight:bold;font-size:14px;margin-bottom:8px'>
  Diagnostic Codes ({len(pipeline_codes)} pipeline codes)
</div>
<table style='border-collapse:collapse;width:100%'>
{''.join(rows_html)}
</table>
</div>
"""
display(HTML(table_html))

In [ ]:
# ════════════════════════════════════════════════════════════
# STRUCTURAL MODES — INVESTIGATION PROMPTS
# ════════════════════════════════════════════════════════════

if sm_codes:
    sm_html = ""
    mode_colors = {
        'SM-BPO-DIS': '#F0883E', 'SM-SCG-INF': '#3FB950', 'SM-MCA-WRN': '#F85149',
        'SM-DMR-DIS': '#FFD700', 'SM-CPL-DIS': '#39D2C0', 'SM-IND-DIS': '#58A6FF',
        'SM-DGN-WRN': '#F85149', 'SM-RTR-DIS': '#39D2C0', 'SM-TNT-DIS': '#3FB950',
        'SM-OVC-CAL': '#BC8CFF',
    }
    for c in sm_codes:
        color = mode_colors.get(c['code'], '#58A6FF')
        sm_html += f"""
        <div style='background:#1c1c2e;padding:10px 14px;margin:6px 0;border-left:3px solid {color};border-radius:4px'>
          <div style='font-family:Consolas;color:{color};font-size:12px;font-weight:bold'>
            ▸ [{c['code']}] {c['short']}
          </div>
          <div style='color:#E6EDF3;font-size:12px;margin-top:4px;line-height:1.5'>
            {c['verbose']}
          </div>
        </div>
        """

    display(HTML(f"""
    <div style='background:#161B22;padding:16px;border-radius:8px;border:2px solid #FFD700;margin:8px 0'>
      <div style='color:#FFD700;font-weight:bold;font-size:16px;margin-bottom:12px'>
        ✦ Structural Modes — Investigation Prompts ({len(sm_codes)} detected)
      </div>
      <div style='color:#8B949E;font-size:11px;margin-bottom:10px'>
        These are second-order diagnoses based on the combination of pipeline codes.
        Each mode suggests a specific investigation path. The tool asks — the expert answers.
      </div>
      {sm_html}
    </div>
    """))
else:
    display(HTML("""
    <div style='background:#161B22;padding:16px;border-radius:8px;margin:8px 0'>
      <div style='color:#8B949E;font-size:14px'>
        No structural modes fired. Data may be minimal (N < 10) or the composition
        is structurally simple.
      </div>
    </div>
    """))

In [ ]:
# ════════════════════════════════════════════════════════════
# FULL REPORT (selected language)
# ════════════════════════════════════════════════════════════

rpt = report(codes, lang=REPORT_LANGUAGE, result=result)

display(HTML(f"""
<details style='background:#161B22;padding:12px;border-radius:8px;margin:8px 0'>
  <summary style='color:#FFD700;font-weight:bold;cursor:pointer;font-size:14px'>
    Full Diagnostic Report ({REPORT_LANGUAGE.upper()}) — click to expand
  </summary>
  <pre style='color:#E6EDF3;font-family:Consolas;font-size:11px;margin-top:8px;white-space:pre-wrap'>{rpt}</pre>
</details>
"""))

In [ ]:
# ════════════════════════════════════════════════════════════
# INSTRUMENT METROLOGY (optional)
# ════════════════════════════════════════════════════════════

if SHOW_METROLOGY:
    try:
        from hs_metrology import run_full_metrology
        print("Running instrument self-evaluation across all experiments...")
        print("(This evaluates the instrument, not your data.)")
        print()
        metro = run_full_metrology()
    except Exception as e:
        print(f"Metrology not available: {e}")
        print("(Metrology requires access to the experiments/ directory.)")
else:
    print("Metrology skipped. Set SHOW_METROLOGY = True to enable.")

In [ ]:
# ════════════════════════════════════════════════════════════
# SAVE RESULTS
# ════════════════════════════════════════════════════════════

result_path = f"{exp_name.replace(' ', '_')}_results.json"
with open(result_path, 'w') as f:
    json.dump(result, f, indent=2, cls=NumpyEncoder)

report_path = f"{exp_name.replace(' ', '_')}_report_{REPORT_LANGUAGE}.txt"
with open(report_path, 'w', encoding='utf-8') as f:
    f.write(rpt)

print(f"Results: {result_path}")
print(f"Report:  {report_path}")
print(f"Dashboard: Hs_dashboard.png")
print()
print("The instrument reads. The expert decides. The loop stays open.")

---

## Pipeline Steps

| Step | Operation | What it produces |
|------|-----------|------------------|
| S1-S3 | Define, identify, load | System identity + raw N×D matrix |
| S4 | Simplex closure | All rows sum to 1.0 |
| S5 | CLR transform | Centred log-ratio coordinates |
| S6 | Aitchison variance | σ²_A trajectory |
| S7 | HVLD vertex lock | Bowl/hill shape + R² |
| S8 | Super squeeze | Transcendental constant matching (35 constants) |
| S9 | EITT entropy | Shannon entropy invariance test |
| S10-S12 | Geometry | Ternary, complex plane, helix embeddings |
| XU | Extended universal | Per-carrier contribution, drift, match density |
| XC | Extended conditional | PID, transfer entropy, ratio lattice |
| SM | Structural modes | Second-order combination analysis → investigation prompts |

---

*Hˢ — Higgins Decomposition on the Simplex*  
*Peter Higgins, 2026*  
*github.com/PeterHiggins19/higgins-decomposition*